# Stage C / NB 15 — LLM reasoner aggregation (E1, E2, E3, E7f)

Protocol reference: experiment families **E1**, **E2**, **E3**, **E7f**; research questions
**RQ6**, **RQ7**. This is the framework proper, and it decides what the paper claims.

## The decisive comparison

NB 14 fitted ridge and gradient-boosted stacking on the same agent outputs and wrote its best
figure to `e7f_target.json`. **E7f must beat that on the same images, with a paired confidence
interval excluding zero**, for an accuracy claim to survive. If it does not, protocol risk K1
applies: the contribution is auditability, not accuracy, and the manuscript should say so
plainly rather than presenting a within-noise difference as a win.

## Execution order is deliberate

Running E1 × E2 × E3 as a cross-product would be 12 × 5 × 5 = 300 configurations. Instead:

1. **E2** — five reasoner identities on the full roster → pick the reasoner.
2. **E3** — five metrics-awareness settings with that reasoner → pick the setting.
3. **E1** — twelve roster configurations with the winning reasoner and setting, all five folds.

That is 5 + 5 + 12 = 22 configurations rather than 300. Steps 1 and 2 select on a nested split
of one fold's inner-validation rows; only step 3 touches held-out folds.

## Two leakage boundaries, not one

The obvious one: reliability shown to the reasoner must not come from the images it is scored
on. NB 13 computes reliability **per fold** — fold *k*'s numbers use only rows flagged
`inner_fold_k`, all of which come from the other four folds.

The subtler one, which the selection stage introduces: if E2 and E3 were both selected on
fold 0's inner-validation rows *and* handed reliability computed on those same rows, the
metrics-aware arms would be reading statistics derived from their own evaluation set. Section 5
therefore splits the selection pool in half by patient group — one half computes reliability,
the other is scored — so the two never overlap.

## E1 and the willingness to delete a module

Each leave-one-out arm removes one agent and reports the paired delta against the full roster.
An agent that does not move either endpoint beyond its confidence interval is reported as
**inconclusive unless its paired interval excludes the equivalence margin**. E1-L6 is the arm referee 1.2 asked for
by name.

## Outputs (under `stage_C/nb15_reasoner/`)
`reasoner_predictions_<config>.jsonl`, `e1_roster_metrics.csv`, `e2_reasoner_metrics.csv`,
`e3_metrics_aware.csv`, `reasoning_traces.jsonl`, `e7f_vs_e7d.json`, `usability.json`,
`gate_nb15.json`.

**Interruption safety.** Every configuration writes one JSON line per image to its own journal,
guarded by a fingerprint of the reasoner, adapter, roster, setting, prompt text and decoding
parameters. Re-run the notebook from the top after any interruption: finished configurations
are skipped, a partial configuration resumes at the next unscored image, and a configuration
whose fingerprint changed is recomputed rather than silently reused.

## 1. Imports and the Stage A path contract

In [ ]:
import gc
import hashlib
import json
import math
import os
import random
import re
import sys
import time
from collections import Counter, OrderedDict, defaultdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch

# Shared metric definitions live with the Stage B notebooks; every arm in Table 2 and every
# fusion/reasoner arm here must be scored by identical code or the comparison is invalid.
_SEARCH = [Path.cwd(), Path.cwd().parent, Path.cwd().parent / "stage_B",
           Path.cwd().parent.parent / "notebooks" / "stage_B"]
for _candidate in _SEARCH:
    if (_candidate / "cxr_metrics.py").is_file():
        sys.path.insert(0, str(_candidate))
        break
else:
    raise FileNotFoundError(f"cxr_metrics.py not found. Searched: {_SEARCH}")
import cxr_metrics as cm

# Stage C's own shared module (prompt rendering, JSON parsing, journals). Kept beside the
# notebooks for the same reason cxr_metrics.py is: two notebooks parsing model output slightly
# differently would show up as a metric difference nobody could explain.
for _candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent / "stage_C",
                   Path.cwd().parent.parent / "notebooks" / "stage_C"]:
    if (_candidate / "stage_c_reasoner.py").is_file():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

SEED = 42
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

FALLBACK_STAGE_A = Path("/data/liangz2/openi/midrc/tetci_resubmit/stage_A")
for _candidate in [FALLBACK_STAGE_A / "nb00_environment" / "stage_a_paths.json",
                   Path.cwd() / "stage_a_paths.json",
                   Path.cwd().parent / "stage_A" / "nb00_environment" / "stage_a_paths.json"]:
    if _candidate.is_file():
        stage_paths = json.loads(_candidate.read_text(encoding="utf-8"))
        print("Path contract:", _candidate)
        break
else:
    raise FileNotFoundError("stage_a_paths.json not found. Run Stage A NB 00 first.")

PROJECT_ROOT = Path(stage_paths["project_root"])
STAGE_ROOT = Path(stage_paths["stage_root"])
STAGE_A_DIR = Path(stage_paths["stage_a_dir"])
STAGE_B_DIR = STAGE_ROOT / "stage_B"
STAGE_C_DIR = STAGE_ROOT / "stage_C"
STAGE_C_DIR.mkdir(parents=True, exist_ok=True)
NB01_DIR = Path(stage_paths["nb_output_dirs"]["nb01_inventory"])
NB02_DIR = Path(stage_paths["nb_output_dirs"]["nb02_folds"])
NB03_DIR = Path(stage_paths["nb_output_dirs"]["nb03_external"])
NB04_DIR = Path(stage_paths["nb_output_dirs"]["nb04_localization"])
FOLD_DEF_DIR = NB02_DIR / "fold_definitions"
MODEL_REVISIONS = stage_paths.get("model_revisions", {})

N_FOLDS = 5
INTERNAL_VALIDATION_FRACTION = 0.10   # identical to Stage B, so inner splits match exactly

print("Stage C output:", STAGE_C_DIR)
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

In [ ]:
# Prompt rendering, constrained generation, JSON parsing and journal handling live in
# stage_c_reasoner.py beside these notebooks, so NB 15 and NB 16 cannot drift apart. Its 58
# self-tests run with `python stage_c_reasoner.py`.
import stage_c_reasoner as scr

print("stage_c_reasoner loaded from", scr.__file__)
print("Message schemas available for image binding:", scr.MESSAGE_SCHEMAS)

## 2. Configuration

In [ ]:
NB15_DIR = STAGE_C_DIR / "nb15_reasoner"
JOURNAL_DIR = NB15_DIR / "journals"
for d in (NB15_DIR, JOURNAL_DIR):
    d.mkdir(parents=True, exist_ok=True)
NB13_DIR = STAGE_C_DIR / "nb13_registry"
NB14_DIR = STAGE_C_DIR / "nb14_fusion"

# E2 candidates. `adapter` is a per-fold template; None means the base model.
REASONER_CANDIDATES = OrderedDict([
    ("R_medgemma_base", {"model_id": "google/medgemma-1.5-4b-it",
                         "loader": "image_text_to_text", "adapter": None}),
    ("R_medgemma_lora", {"model_id": "google/medgemma-1.5-4b-it",
                         "loader": "image_text_to_text",
                         "adapter": str(STAGE_B_DIR / "nb09_medgemma_lora" / "folds"
                                        / "fold_{fold}" / "best_adapter")}),
    ("R_qwen_base",     {"model_id": "Qwen/Qwen3.5-4B", "loader": "multimodal_lm",
                         "adapter": None, "disable_thinking": True}),
    ("R_qwen_lora",     {"model_id": "Qwen/Qwen3.5-4B", "loader": "multimodal_lm",
                         "adapter": str(STAGE_B_DIR / "nb10_qwen_lora" / "folds"
                                        / "fold_{fold}" / "best_adapter"),
                         "disable_thinking": True}),
    ("R_nvreason",      {"model_id": "nvidia/NV-Reason-CXR-3B", "loader": "auto",
                         "adapter": None}),
])

E1_ADD_IN = ["A2", "A3", "A4", "A5", "A6"]   # add-one-in order for the E1 ladder
E3_SETTINGS = ["E3a", "E3b", "E3c", "E3d", "E3e"]

SELECTION_FOLD = 0            # E2/E3 are selected on this fold's inner-validation rows
SELECTION_MAX_IMAGES = 300    # cap per selection configuration; 5+5 configs at this size
RUN_E2, RUN_E3, RUN_E1 = True, True, True
RUN_INNER_THRESHOLD_SCORES = True  # required by NB 18; resumable, full roster only
E1_FOLDS = [0, 1, 2, 3, 4]
E1_MAX_IMAGES_PER_FOLD = None  # None = every test image; set an int for a cheaper first pass

MAX_NEW_TOKENS = 384
USE_JSON_STOP_CRITERIA = True
CONFLICT_THRESHOLD_QUANTILE = 0.90   # E3e flags disagreement above this fitting-set quantile
N_BOOTSTRAP = 2000
FLUSH_EVERY = 25              # journal is append-per-item; this only controls progress printing
MAX_SESSION_HOURS = 35.0      # Biowulf limit is 36 h; stop cleanly between images
SESSION_DEADLINE = scr.make_session_deadline(MAX_SESSION_HOURS)

SMOKE_TEST = False            # True -> 16 images per configuration, to verify wiring
if SMOKE_TEST:
    SELECTION_MAX_IMAGES, E1_MAX_IMAGES_PER_FOLD = 16, 16

# Forcing recomputation, per RESUME.md
FORCE_RECOMPUTE_CONFIGS = []  # e.g. ["E2_R_qwen_base"] -- journals are deleted and re-run
FORCE_RECOMPUTE_ALL = False

print("E2 candidates:", list(REASONER_CANDIDATES))
print("E3 settings  :", E3_SETTINGS)
print(f"Soft stop    : {MAX_SESSION_HOURS:.1f} h after this cell")
print("Output       :", NB15_DIR)

## 3. Load the registry, its fold-relative membership, and NB 14's target

Three things are verified rather than trusted, because each has already gone wrong once in this
project: that NB 13 certified its reliability table as inner-validation only; that the
fold-relative membership columns exist (a single `split` column cannot express "inner validation
*for fold k*", and a notebook built on one silently produces empty fitting sets); and that
NB 14 actually ran, since without its target E7f has nothing to beat.

In [ ]:
registry_path = NB13_DIR / "agent_registry.parquet"
if registry_path.is_file():
    try:
        registry = pd.read_parquet(registry_path)
    except Exception:
        registry = pd.read_csv(NB13_DIR / "agent_registry.csv")
elif (NB13_DIR / "agent_registry.csv").is_file():
    registry = pd.read_csv(NB13_DIR / "agent_registry.csv")
else:
    raise FileNotFoundError(
        f"No registry at {NB13_DIR}. Run NB 13 first — it is what guarantees the reliability "
        "statistics handed to the reasoner were computed on inner validation only.")

reliability_table = pd.read_csv(NB13_DIR / "agent_reliability_inner.csv")
manifest = json.loads((NB13_DIR / "registry_manifest.json").read_text(encoding="utf-8"))
if manifest.get("reliability_scope") != "inner_validation_only":
    raise RuntimeError(
        "NB 13 did not certify its reliability table as inner-validation only. E3 would be "
        "circular: the reasoner would be handed metrics derived from the images it is scored "
        "on. Re-run NB 13 and confirm its leakage assertion passes.")

INNER_FOLD_COLUMNS = [f"inner_fold_{k}" for k in range(N_FOLDS)]
missing_columns = [c for c in INNER_FOLD_COLUMNS if c not in registry.columns]
if missing_columns:
    raise RuntimeError(
        f"The registry lacks {missing_columns}. Re-run NB 13: inner-validation membership is a "
        "relation between an image and a fold, not a property of the image, and without the "
        "per-fold columns every reliability set here would be empty.")
print("Reliability provenance verified:", manifest["reliability_scope"],
      "| indexing:", manifest.get("reliability_indexing"))

SELECTED_FOLD_COLUMNS = [f"selected_for_fold_{k}" for k in range(N_FOLDS)]
missing_selection = [c for c in SELECTED_FOLD_COLUMNS if c not in registry.columns]
if missing_selection:
    raise RuntimeError(f"Registry lacks {missing_selection}; rerun revised NB 13.")
AGENTS = [a for a in ["A1", "A2", "A3", "A4", "A5", "A6"]
          if a in set(registry["agent"])]
if len(AGENTS) < 2:
    raise RuntimeError(f"The framework needs at least two agents; found {AGENTS}. "
                       "Run more Stage B notebooks first.")

meta = (registry.drop_duplicates("image_key").set_index("image_key")
        [["fold", "group_id", "gt_mrale_total", "gt_covid", "severity_band"]
         + INNER_FOLD_COLUMNS])
AGENT_ROWS_BY_FOLD, FRAME_BY_FOLD = {}, {}
UNC_BY_FOLD, COVID_PRED_BY_FOLD, COVID_SCORE_BY_FOLD = {}, {}, {}
for fold in range(N_FOLDS):
    chosen = registry[(registry[f"selected_for_fold_{fold}"] == True)
                      & registry["agent"].isin(AGENTS)].copy()
    AGENT_ROWS_BY_FOLD[fold] = chosen
    presence = chosen.pivot_table(index="image_key", columns="agent",
                                  values="arm", aggfunc="size", fill_value=0)
    for agent in AGENTS:
        if agent not in presence.columns:
            presence[agent] = 0
    eligible = presence.index[(presence[AGENTS] > 0).all(axis=1)]
    wide = chosen.pivot_table(index="image_key", columns="agent",
                              values="mrale_total", aggfunc="first").reindex(
                                  columns=AGENTS)
    FRAME_BY_FOLD[fold] = meta.join(wide, how="inner").loc[eligible]
    UNC_BY_FOLD[fold] = chosen.pivot_table(index="image_key", columns="agent",
                                            values="mrale_uncertainty", aggfunc="first")
    COVID_PRED_BY_FOLD[fold] = chosen.pivot_table(
        index="image_key", columns="agent", values="covid_pred", aggfunc="first")
    COVID_SCORE_BY_FOLD[fold] = chosen.pivot_table(
        index="image_key", columns="agent", values="covid_score", aggfunc="first")

frame = pd.concat([FRAME_BY_FOLD[k][FRAME_BY_FOLD[k]["fold"] == k]
                   for k in range(N_FOLDS)]).sort_index()
if not frame.index.is_unique:
    raise RuntimeError("Fold-specific framework denominator contains duplicate images.")
print(f"\nAgents: {AGENTS}")
print(f"Images scored by EVERY agent (the framework denominator): {len(frame):,}")
print("  Every E1 arm is scored on this same set, so leave-one-out deltas are paired.")
for k in range(N_FOLDS):
    fold_frame = FRAME_BY_FOLD[k]
    print(f"  fold {k}: test {int((fold_frame['fold'] == k).sum()):,} | "
          f"reliability rows {int((fold_frame[f'inner_fold_{k}'] == 1).sum()):,}")

e7f_target = {}
target_path = NB14_DIR / "e7f_target.json"
if target_path.is_file():
    e7f_target = json.loads(target_path.read_text(encoding="utf-8"))
    print(f"\nNB 14 target: {e7f_target.get('best_stacking_arm')} "
          f"MAE {e7f_target.get('best_stacking_mae')}")
else:
    print("\nWARNING: NB 14 has not run, so E7f has no baseline to beat and the framework's "
          "central claim cannot be evaluated. Run NB 14 before reporting E7f.")

views = pd.read_csv(NB04_DIR / "view_index.csv").set_index("image_key")
def preferred_image_path(record):
    for column in ["v0_image", "v1_thorax_image"]:
        value = record.get(column)
        if isinstance(value, str) and value.strip() and Path(value).is_file():
            return value
    return None

IMAGE_PATHS = {str(k): preferred_image_path(r)
               for k, r in views.to_dict("index").items()}
missing_paths = [k for k in frame.index if not IMAGE_PATHS.get(str(k))]
print(f"Image paths resolved: {len(frame) - len(missing_paths):,}/{len(frame):,} "
      "(V0 whole radiograph preferred, V1 fallback)")
if missing_paths:
    raise FileNotFoundError(
        f"{len(missing_paths):,} framework images have neither a readable V0 image nor a "
        f"readable V1 fallback (examples: {missing_paths[:5]}). Re-run/check NB 04; silently "
        "dropping these cases would change the registered denominator.")

evidence = defaultdict(dict)
for label, path in [("A6", STAGE_B_DIR / "nb08_biomedclip_entity" / "entity_findings.jsonl"),
                    ("A4", STAGE_B_DIR / "nb11_nvreason" / "nvreason_findings.jsonl")]:
    if path.is_file():
        for row in cm.read_jsonl(path):
            evidence[str(row.get("image_key"))][label] = row
print(f"Qualitative evidence channels loaded for {len(evidence):,} images")

## 4. Build the E1 roster grid from the agents that actually exist

The protocol's grid assumes A1–A6. Stage B may be partially complete, so the grid is built from
what is present and the manuscript is told which roster was ablated. Claiming a full-roster
ablation over a partial roster would misstate the result.

In [ ]:
present = [a for a in ["A1", "A2", "A3", "A4", "A5", "A6"] if a in AGENTS]
extra_agents = [a for a in AGENTS if a not in present]
add_in = [a for a in E1_ADD_IN if a in present]

candidate_rosters = OrderedDict()
candidate_rosters["E1-0_reasoner_only"] = []
for i in range(1, len(add_in) + 1):
    candidate_rosters[f"E1-{i}_add_{add_in[i - 1]}"] = add_in[:i]
candidate_rosters["E1-full"] = present
for agent in present:
    candidate_rosters[f"E1-L_minus_{agent}"] = [a for a in present if a != agent]

# Deduplicate. When an agent is absent the add-one-in ladder collides with the full roster and
# with a leave-one-out arm -- with A1 missing, `E1-4_add_A6`, `E1-full` and `E1-3_add_A5`,
# `E1-L_minus_A6` are the same roster. Generating each twice would burn GPU-hours to produce
# two identical columns, and would invite reading the sampling noise between them as an effect.
def _name_priority(name):
    """E1-full and the leave-one-out arms carry the analytic meaning, so they name the run."""
    if name == "E1-full":
        return 0
    if "minus" in name:
        return 1
    if name.startswith("E1-0"):
        return 2
    return 3


by_signature = OrderedDict()
for name, roster in candidate_rosters.items():
    by_signature.setdefault(tuple(sorted(roster)), []).append(name)

E1_ROSTERS, E1_ALIASES = OrderedDict(), {}
for signature, names in by_signature.items():
    canonical = sorted(names, key=lambda n: (_name_priority(n), n))[0]
    E1_ROSTERS[canonical] = candidate_rosters[canonical]
    aliases = [n for n in names if n != canonical]
    if aliases:
        E1_ALIASES[canonical] = aliases

print(f"Roster present : {present}")
if extra_agents:
    print(f"Also available : {extra_agents} (kept out of the E1 ladder; they are not "
          "protocol agents)")
print(f"E1 configurations: {len(E1_ROSTERS)}")
for name, roster in E1_ROSTERS.items():
    alias = E1_ALIASES.get(name)
    print(f"  {name:<24} {roster or '(image only, no agent assistance)'}"
          + (f"   [also the {', '.join(alias)} arm; run once]" if alias else ""))

absent = sorted({"A1", "A2", "A3", "A4", "A5", "A6"} - set(present))
if absent:
    print()
    print(f"NOT PRESENT: {absent}. The manuscript must report the roster actually ablated.")
    if "A6" in absent:
        print("  E1-L6 — the arm referee 1.2 asked for by name — needs A6 and cannot run.")

FULL_ROSTER_ARM = next(name for name, roster in E1_ROSTERS.items()
                       if sorted(roster) == sorted(present))
FULL_ROSTER = E1_ROSTERS[FULL_ROSTER_ARM]
if FULL_ROSTER_ARM != "E1-full":
    print(f"\nThe full-roster arm is named {FULL_ROSTER_ARM} after deduplication.")

## 5. Reliability contexts — and the nested split that keeps selection honest

Reliability is recomputed here from the registry rather than read from NB 13's CSV, then
cross-checked against it. Two reasons: the reasoner needs a dictionary keyed by agent at prompt
time, and the selection stage needs a context NB 13 does not produce.

**Fold contexts (used by E1).** Fold *k*'s reliability comes from rows flagged `inner_fold_k`,
which by construction are from the other four folds.

**The selection context (used by E2 and E3).** E2 and E3 are chosen on fold 0's
inner-validation rows. If those same rows also produced the reliability numbers in the prompt,
a metrics-aware setting would be reading statistics computed on its own evaluation set — it
could look good for a reason that will not generalise. So the pool is split in half **by
patient group**: one half computes reliability, the other is scored. Grouping matters because
two radiographs of the same patient are not independent.

In [ ]:
def compute_reliability(keys, fold):
    """Per-agent reliability over exactly `keys`. Mirrors NB 13's definitions."""
    keys = set(map(str, keys))
    subset = AGENT_ROWS_BY_FOLD[fold][
        AGENT_ROWS_BY_FOLD[fold]["image_key"].astype(str).isin(keys)]
    out = {}
    for agent, group in subset.groupby("agent"):
        rows = group.to_dict("records")
        entry = {"n": len(rows)}
        mrale_rows = [{"gt_mrale_total": r["gt_mrale_total"], "mrale_total": r["mrale_total"]}
                      for r in rows if r.get("gt_mrale_total") is not None]
        if mrale_rows:
            m = cm.mrale_metrics(mrale_rows)
            entry["mrale_mae"] = float(m["mae"])
            entry["mrale_qwk"] = float(m.get("qwk", float("nan")))
        covid_rows = [r for r in rows if r.get("gt_covid") in {"Yes", "No"}]
        if covid_rows:
            c = cm.classification_metrics(
                [r["gt_covid"] for r in covid_rows], [r.get("covid_pred") for r in covid_rows],
                [r.get("covid_score") for r in covid_rows])
            if c.get("auroc") is not None:
                entry["covid_auroc"] = float(c["auroc"])
            scored = [(1 if r["gt_covid"] == "Yes" else 0, r.get("covid_score"))
                      for r in covid_rows if r.get("covid_score") is not None]
            if len(scored) > 30 and len({y for y, _ in scored}) == 2:
                probability = np.clip(np.asarray([p for _, p in scored], dtype=float),
                                      1e-6, 1 - 1e-6)
                logit = np.log(probability / (1 - probability)).reshape(-1, 1)
                if logit.std() > 1e-9:
                    from sklearn.linear_model import LogisticRegression
                    y = np.asarray([y for y, _ in scored], dtype=int)
                    try:
                        calibration = LogisticRegression(penalty=None, max_iter=2000).fit(logit, y)
                    except (TypeError, ValueError):
                        calibration = LogisticRegression(penalty="none", max_iter=2000).fit(logit, y)
                    entry["covid_calibration_slope"] = float(calibration.coef_[0, 0])
        out[agent] = entry
    return out


def conflict_threshold(keys, fold):
    fold_frame = FRAME_BY_FOLD[fold]
    subset = fold_frame.loc[[k for k in map(str, keys) if k in fold_frame.index]]
    if len(subset) < 20:
        return float("inf")
    spreads = subset[AGENTS].astype(float).std(axis=1, skipna=True)
    return float(np.nanquantile(spreads, CONFLICT_THRESHOLD_QUANTILE))


def make_context(label, fit_keys, fold):
    rel = compute_reliability(fit_keys, fold)
    ctx = {"label": label, "reliability": rel, "n_fit": len(set(map(str, fit_keys))),
           "conflict_threshold": conflict_threshold(fit_keys, fold), "fold": fold}
    return ctx


# ---- Fold contexts, for E1 ----------------------------------------------------------------
FOLD_CONTEXT = {}
for k in range(N_FOLDS):
    fold_frame = FRAME_BY_FOLD[k]
    fit_keys = fold_frame.index[fold_frame[f"inner_fold_{k}"] == 1]
    FOLD_CONTEXT[k] = make_context(f"fold_{k}", fit_keys, k)
    maes = {a: round(v["mrale_mae"], 3) for a, v in FOLD_CONTEXT[k]["reliability"].items()
            if "mrale_mae" in v}
    print(f"  fold {k}: reliability from {FOLD_CONTEXT[k]['n_fit']:,} rows  MAE {maes}")

# ---- Selection context, for E2 and E3 -----------------------------------------------------
selection_frame = FRAME_BY_FOLD[SELECTION_FOLD]
pool = selection_frame[selection_frame[f"inner_fold_{SELECTION_FOLD}"] == 1]
pool_groups = sorted(pool["group_id"].astype(str).unique())
rng = random.Random(SEED + 9001)
rng.shuffle(pool_groups)
half = len(pool_groups) // 2
fit_groups, eval_groups = set(pool_groups[:half]), set(pool_groups[half:])
SELECTION_FIT_KEYS = [str(k) for k in
                      pool.index[pool["group_id"].astype(str).isin(fit_groups)]]
SELECTION_EVAL = pool[pool["group_id"].astype(str).isin(eval_groups)]
if SELECTION_MAX_IMAGES and len(SELECTION_EVAL) > SELECTION_MAX_IMAGES:
    SELECTION_EVAL = SELECTION_EVAL.sample(n=SELECTION_MAX_IMAGES, random_state=SEED)
SELECTION_EVAL = SELECTION_EVAL.sort_index()
SELECTION_CONTEXT = make_context("selection", SELECTION_FIT_KEYS, SELECTION_FOLD)

overlap = set(SELECTION_FIT_KEYS) & set(map(str, SELECTION_EVAL.index))
if overlap:
    raise RuntimeError(
        f"{len(overlap)} images are in BOTH halves of the selection split. E2 and E3 would be "
        "selected using reliability computed on their own evaluation rows.")
group_overlap = fit_groups & eval_groups
if group_overlap:
    raise RuntimeError(f"{len(group_overlap)} patient groups span both halves of the "
                       "selection split; same-patient images are not independent.")
print()
print(f"Selection pool (fold {SELECTION_FOLD} inner validation): {len(pool):,} images, "
      f"{len(pool_groups):,} groups")
print(f"  reliability half: {len(SELECTION_FIT_KEYS):,} images / {len(fit_groups):,} groups")
print(f"  evaluation half : {len(SELECTION_EVAL):,} images / {len(eval_groups):,} groups")
print("  disjoint by image and by patient group — verified above")

# Cross-check against NB 13's own table, which used the whole fold-k set.
nb13_for_fold = reliability_table[reliability_table.get("scope") == "inner_validation_for_fold"] \
    if "scope" in reliability_table.columns else pd.DataFrame()
if len(nb13_for_fold) and "for_fold" in nb13_for_fold.columns:
    print()
    print("Cross-check against NB 13 (fold 0 reliability, should match to rounding):")
    for row in nb13_for_fold[nb13_for_fold["for_fold"] == 0].to_dict("records"):
        mine = FOLD_CONTEXT[0]["reliability"].get(row["agent"], {}).get("mrale_mae")
        if mine is not None and pd.notna(row.get("mrale_mae")):
            flag = "ok" if abs(mine - float(row["mrale_mae"])) < 0.01 else "MISMATCH"
            print(f"  {row['agent']}: here {mine:.4f} vs NB13 {float(row['mrale_mae']):.4f}"
                  f"  [{flag}]")

## 6. The evidence block — what the reasoner sees, and the E3 ladder

Five settings, each strictly adding to the one before. That nesting is what makes E3
interpretable: a difference between adjacent rungs is attributable to exactly one ingredient.

| setting | adds |
| --- | --- |
| E3a | agent predictions only, no reliability information |
| E3b | + each agent's inner-validation MAE / AUROC |
| E3c | + per-case uncertainty, where the agent reports one |
| E3d | + an explicit reliability ranking and an instruction to weight by it |
| E3e | + a conflict flag when agent spread exceeds the fitting-set 90th percentile |

Every number in the block is recorded alongside the prediction and hashed, so
"agent-output provenance is recoverable for every final answer" is verified by recomputation
rather than asserted.

**On E3d's instruction.** Telling a model to weight by reliability is not the same as it doing
so. Section 12 checks whether E3d's answers actually move toward the more reliable agents. An
instruction that changes nothing is a null result and should be reported as one.

In [ ]:
AGENT_DESCRIPTION = {
    "A1": "anatomy-aware per-lung reader (localises each lung, then scores it)",
    "A2": "MedGemma-1.5-4B fine-tuned with LoRA on this cohort",
    "A3": "Qwen3.5-4B fine-tuned with LoRA on this cohort",
    "A4": "NV-Reason-CXR-3B, a reasoning model that reports findings and may abstain",
    "A5": "CXformer frozen encoder with an ordinal probe",
    "A6": "BiomedCLIP contrastive entity scorer over 22 radiographic findings",
    "E0g": "conventional CNN/ViT classifier trained end-to-end",
}

# The system prompt and response schema are the module's, so NB 16's prompt-variant study
# measures deviations from exactly the prompt NB 15 used.
SYSTEM_PROMPT = scr.DEFAULT_SYSTEM_PROMPT
RESPONSE_SCHEMA = scr.DEFAULT_RESPONSE_SCHEMA


def _finite(value):
    return value is not None and not (isinstance(value, float) and math.isnan(value))


def agent_lines(image_key, roster, setting, ctx):
    lines, provenance = [], {}
    fold = int(ctx["fold"])
    row = FRAME_BY_FOLD[fold].loc[image_key]
    unc = UNC_BY_FOLD[fold]
    covid_pred = COVID_PRED_BY_FOLD[fold]
    covid_score = COVID_SCORE_BY_FOLD[fold]
    selected_rows = AGENT_ROWS_BY_FOLD[fold]
    for agent in roster:
        matches = selected_rows[(selected_rows["image_key"].astype(str) == str(image_key))
                                & (selected_rows["agent"] == agent)]
        source = matches.iloc[0].to_dict() if len(matches) else {}
        value = row.get(agent)
        entry, facts = {}, []
        if _finite(value):
            entry["mrale_total"] = float(value)
            facts.append(f"mRALE {float(value):.0f}")
            right, left = source.get("mrale_right"), source.get("mrale_left")
            if _finite(right) and _finite(left):
                facts.append(f"right {float(right):.0f}, left {float(left):.0f}")
                entry.update({"mrale_right": float(right), "mrale_left": float(left)})
        rel = ctx["reliability"].get(agent, {})
        # E3b = reliability only; E3c = case uncertainty only; E3d/e combine both.
        if setting in {"E3b", "E3d", "E3e"}:
            bits = []
            if _finite(rel.get("mrale_mae")):
                bits.append(f"validation MAE {rel['mrale_mae']:.2f}")
                entry["shown_mae"] = round(float(rel["mrale_mae"]), 4)
            if _finite(rel.get("covid_auroc")):
                bits.append(f"validation AUROC {rel['covid_auroc']:.3f}")
                entry["shown_auroc"] = round(float(rel["covid_auroc"]), 4)
            if _finite(rel.get("covid_calibration_slope")):
                bits.append(f"calibration slope {rel['covid_calibration_slope']:.2f}")
                entry["shown_calibration_slope"] = round(
                    float(rel["covid_calibration_slope"]), 4)
            if bits:
                facts.append("reliability: " + "; ".join(bits))
        if setting in {"E3c", "E3d", "E3e"} and agent in unc.columns \
                and image_key in unc.index:
            uncertainty_value = unc.at[image_key, agent]
            if _finite(uncertainty_value):
                facts.append(f"case uncertainty {float(uncertainty_value):.2f}")
                entry["case_uncertainty"] = round(float(uncertainty_value), 4)
        if agent in covid_pred.columns and image_key in covid_pred.index:
            call = covid_pred.at[image_key, agent]
            if isinstance(call, str) and call in {"Yes", "No"}:
                score = covid_score.at[image_key, agent] if (agent in covid_score.columns
                                                           and image_key in covid_score.index) else None
                facts.append(f"COVID {call}" + (f" (P+ {float(score):.3f})"
                                                   if _finite(score) else ""))
                entry["covid_pred"] = call
                if _finite(score):
                    entry["covid_score"] = round(float(score), 6)
        finding_row = evidence.get(str(image_key), {}).get(agent, {})
        findings = finding_row.get("findings") or []
        if isinstance(findings, str):
            findings = [findings]
        findings = [str(item) for item in findings[:8]]
        if findings:
            facts.append("findings: " + ", ".join(findings))
            entry["findings"] = findings
        if not facts:
            facts.append("no usable numeric output; agent abstained")
            entry["abstained"] = True
        lines.append(f"- {agent} ({AGENT_DESCRIPTION.get(agent, 'agent')}): "
                     + "; ".join(facts))
        provenance[agent] = entry
    return lines, provenance


def evidence_block(image_key, roster, setting, ctx):
    lines, provenance = agent_lines(image_key, roster, setting, ctx)
    meta = {"agents": provenance, "setting": setting, "roster": list(roster),
            "context": ctx["label"]}
    if not lines:
        return ("No agent outputs are available for this image; judge from the radiograph "
                "alone."), meta

    parts = ["Agent reports for this radiograph:"] + lines

    if setting in {"E3d", "E3e"}:
        ranked = [a for a in roster
                  if _finite(ctx["reliability"].get(a, {}).get("mrale_mae"))]
        ranked.sort(key=lambda a: ctx["reliability"][a]["mrale_mae"])
        if ranked:
            parts += ["",
                      "Reliability ranking on held-out validation data, most accurate first: "
                      + " > ".join(ranked) + ".",
                      "Weight the agents accordingly. Where they disagree, prefer the more "
                      "reliable one unless the image clearly contradicts it."]
            meta["reliability_ranking"] = ranked

    if setting == "E3e":
        values = [v["mrale_total"] for v in provenance.values()
                  if _finite(v.get("mrale_total"))]
        spread = float(np.std(values)) if len(values) > 1 else 0.0
        meta["agent_spread"] = round(spread, 4)
        meta["conflict_flagged"] = bool(spread >= ctx["conflict_threshold"])
        if meta["conflict_flagged"]:
            parts += ["",
                      f"CONFLICT: the agents disagree unusually strongly here (spread "
                      f"{spread:.1f}, above the {CONFLICT_THRESHOLD_QUANTILE:.0%} validation "
                      "percentile). Re-read the image yourself rather than averaging, and say "
                      "in your rationale which agent you sided with."]
    return "\n".join(parts), meta


def user_prompt_for(image_key, roster, setting, ctx):
    if roster:
        block, meta = evidence_block(image_key, roster, setting, ctx)
        prompt = ("Independent automated agents have already read this radiograph. Their "
                  "reports are below. Look at the image yourself, then produce ONE final "
                  "answer. You may override any agent; if you do, say so in the rationale.\n\n"
                  f"{block}\n\nReturn exactly this JSON:\n{RESPONSE_SCHEMA}")
    else:
        meta = {"agents": {}, "setting": setting, "roster": [], "context": ctx["label"]}
        prompt = ("Score this radiograph yourself, with no agent assistance.\n\n"
                  f"Return exactly this JSON:\n{RESPONSE_SCHEMA}")
    meta["prompt_sha256"] = hashlib.sha256(prompt.encode("utf-8")).hexdigest()[:16]
    return prompt, meta


_demo_key = str(SELECTION_EVAL.index[0]) if len(SELECTION_EVAL) else str(frame.index[0])
_demo_prompt, _demo_meta = user_prompt_for(_demo_key, FULL_ROSTER, "E3e", SELECTION_CONTEXT)
print("-" * 78)
print(_demo_prompt)
print("-" * 78)
print("provenance record stored with every answer:")
print(json.dumps(_demo_meta, indent=2)[:800])

## 7. Generation and parsing come from the shared module

`stage_c_reasoner.py` owns prompt rendering, constrained generation, and answer parsing. Three
of its behaviours matter enough to state here rather than leave in the source:

**Image binding is verified, not assumed.** `render_prompt` tries six message schemas until one
produces the model's image placeholder. `{"type": "image"}` with no value suffices for MedGemma
and Qwen3.5, but Qwen2.5-VL-derived templates (NV-Reason-CXR-3B) emit nothing for it: the
prompt then carries no vision token, the image is never attended to, and the model politely
asks for a radiograph while every parse fails. That is what produced a 20-hour NB 07 run at
`valid_rate = 0.000`.

**Generation stops when the first JSON object closes.** Otherwise these models emit the same
object several times plus a stray control token — parsing recovers the first copy, so metrics
look fine while token counts and latency measure repetition. That is protocol item E6-Q, and
NB 16 depends on it being fixed.

**Unparseable output is invalid, not missing.** `cxr_metrics` then applies the same fixed
24-point penalty every other arm receives, so an arm cannot buy accuracy by declining to answer
the hard cases. `agents_used` is checked against the roster actually supplied: a model citing an
agent it was never shown is fabricating provenance, and section 14 counts those.

In [ ]:
# Which message schema each model needed, filled in on first use and reported in run_config.
IMAGE_TEMPLATE_VARIANT = {}


def generate_json(model, processor, spec, image_path, user_prompt):
    """Greedy generation for the primary results. NB 16 varies temperature and top_p."""
    text, n_completion, n_prompt, n_objects = scr.generate_json(
        model, processor, spec, image_path, user_prompt, IMAGE_TEMPLATE_VARIANT,
        system_prompt=SYSTEM_PROMPT, max_new_tokens=MAX_NEW_TOKENS,
        temperature=0.0, use_json_stop=USE_JSON_STOP_CRITERIA)
    return text, n_completion, n_prompt


def load_reasoner(label, spec, fold):
    return scr.load_reasoner(label, spec, fold, MODEL_REVISIONS)


parse_reasoner_output = scr.parse_reasoner_output
release = scr.release

# Confirm the module behaves as this notebook assumes before any GPU time is spent.
_fields, _error, _notes = parse_reasoner_output(
    '{"extent_right": 2, "density_right": 2, "extent_left": 2, '
    '"density_left": 1, "mrale_right": 4, "mrale_left": 2, '
    '"mrale_total": 6, "covid_positive": "No", "covid_confidence": 0.9, '
    '"agents_used": ["A2", "A9"]}', roster=["A2"])
assert _error is None and _fields["mrale_total"] == 6, "parser contract changed"
assert abs(_fields["covid_score"] - 0.1) < 1e-9, "covid score must be P(positive)"
assert _notes["agents_fabricated"] == ["A9"], "fabricated citations must be detected"
print("Parser contract verified against stage_c_reasoner.")

## 9. The run engine, and what survives an interruption

One journal per configuration, one JSON line per image, appended as it is produced. Re-running
the notebook re-reads the journal and scores only what is missing, so the most an interruption
costs is **one image**.

Three details that this project learned the hard way:

1. **The resume key is a tuple.** `cm.load_jsonl_by_key` returns tuple keys; comparing a bare
   string against them never matches, which prints a reassuring "resuming with N cached" and
   then rescores everything. NB 11 shipped with that bug. The lookup below builds
   `(str(image_key),)`.
2. **The fingerprint covers everything that changes the answer** — reasoner, adapter path,
   roster, E3 setting, both prompt texts, decoding parameters, and the reliability numbers
   themselves. Change a prompt phrase and the affected journal is invalidated instead of
   silently returning stale answers. NB 08's entity cache omitted the phrases and made a
   correct fix look like it had no effect.
3. **Models are loaded once per (reasoner, fold)**, not once per configuration. With twelve E1
   rosters that is the difference between 5 and 60 model loads.

In [ ]:
def config_fingerprint(job):
    prompt_identity = []
    for key in sorted(map(str, job["rows"].index)):
        _, provenance = user_prompt_for(key, job["roster"], job["setting"],
                                        job["context"])
        prompt_identity.append((key, provenance["prompt_sha256"], IMAGE_PATHS.get(key)))
    payload = {
        "reasoner": job["reasoner"],
        "model_id": job["spec"]["model_id"],
        "adapter": (str(job["spec"].get("adapter")).format(fold=job["fold"])
                    if job["spec"].get("adapter") else None),
        "loader_spec": job["spec"],
        "revision": MODEL_REVISIONS.get(job["spec"]["model_id"]),
        "roster": list(job["roster"]),
        "setting": job["setting"],
        "system_prompt": SYSTEM_PROMPT,
        "response_schema": RESPONSE_SCHEMA,
        "max_new_tokens": MAX_NEW_TOKENS,
        "json_stop": USE_JSON_STOP_CRITERIA,
        "greedy": True, "image_view": "v0_whole_with_v1_fallback",
        "token_score_probe": scr.TOKEN_SCORE_PROBE_VERSION,
        "prompt_identity": scr.fingerprint(prompt_identity),
        # The reliability numbers are part of the prompt, so they are part of the identity.
        "reliability": {a: {k: v for k, v in entry.items() if k != "n"}
                        for a, entry in sorted(job["context"]["reliability"].items())},
        "conflict_threshold": (None if not math.isfinite(job["context"]["conflict_threshold"])
                               else round(job["context"]["conflict_threshold"], 6)),
    }
    # Preserve the fingerprints of completed outer-test journals. Only the new NB 18
    # threshold-selection jobs add an output contract to their identity.
    if (job.get("score_role", "outer_test") != "outer_test"
            or job.get("output_arm", job["name"]) != job["name"]):
        payload["output_contract"] = {
            "output_arm": job.get("output_arm", job["name"]),
            "score_role": job.get("score_role", "outer_test")}
    return scr.fingerprint(payload)


def journal_name(job):
    """
    One journal per (configuration, FOLD).

    The fingerprint includes the reliability numbers, and those are fold-specific by design --
    fold k's come from rows flagged inner_fold_k. A single journal per configuration therefore
    looks "changed" at every fold boundary and retires the previous fold's work: five folds run,
    one fold survives, and the arm silently reports a fifth of its data with a plausible-looking
    MAE. Keying the journal by fold keeps each fold's cache valid on its own terms.
    """
    return f"{job['name']}__fold{job['fold']}"


def journal_for(name, config_fingerprint, config_name=None):
    """Return (journal_path, already-scored image keys). Invalidates on fingerprint change."""
    force = (FORCE_RECOMPUTE_ALL or name in FORCE_RECOMPUTE_CONFIGS
             or (config_name is not None and config_name in FORCE_RECOMPUTE_CONFIGS))
    path, _ = scr.open_journal(JOURNAL_DIR, name, config_fingerprint, force=force,
                               log=lambda message: print("  " + str(message).lstrip()))
    return path, scr.journal_keys(path)


def make_job(name, reasoner, roster, setting, fold, rows, ctx, *,
             output_arm=None, score_role="outer_test"):
    return {"name": name, "reasoner": reasoner, "spec": {**REASONER_CANDIDATES[reasoner],
                                                         "_label": reasoner},
            "roster": list(roster), "setting": setting, "fold": fold, "rows": rows,
            "context": ctx, "output_arm": output_arm or name,
            "score_role": score_role}


def run_job(job, model, processor):
    fingerprint = config_fingerprint(job)
    path, done = journal_for(journal_name(job), fingerprint, job["name"])
    rows = job["rows"]
    pending = [k for k in map(str, rows.index) if k not in done]
    print(f"  {job['name']} fold {job['fold']}: {len(rows):,} images, "
          f"{len(rows) - len(pending):,} cached, {len(pending):,} pending")
    if not pending:
        return path

    started = time.perf_counter()
    for index, key in enumerate(pending, start=1):
        scr.check_session_deadline(
            SESSION_DEADLINE, f"NB 15 {job['name']} fold {job['fold']}")
        truth = FRAME_BY_FOLD[int(job["fold"])].loc[key]
        prompt, provenance = user_prompt_for(key, job["roster"], job["setting"],
                                             job["context"])
        began = time.perf_counter()
        try:
            text, n_tokens, prompt_tokens = generate_json(
                model, processor, job["spec"], IMAGE_PATHS[key], prompt)
            fields, parse_error, notes = parse_reasoner_output(text, job["roster"])
            self_reported_decision = fields.get("covid_pred")
            self_reported_score = fields.get("covid_score")
            try:
                token_score, token_candidates = scr.covid_token_probability(
                    model, processor, job["spec"], IMAGE_PATHS[key], prompt,
                    IMAGE_TEMPLATE_VARIANT, system_prompt=SYSTEM_PROMPT)
            except Exception as scoring_exc:
                token_score, token_candidates = None, {}
                notes["covid_token_scoring_error"] = (
                    f"{type(scoring_exc).__name__}: {scoring_exc}")
            notes["covid_self_reported_score"] = self_reported_score
            notes["covid_self_reported_decision"] = self_reported_decision
            notes["covid_candidate_log_likelihoods"] = token_candidates
            fields["covid_score"] = token_score
            if token_score is not None:
                fields["covid_pred"] = "Yes" if token_score >= 0.5 else "No"
        except Exception as exc:
            # Record the failure and continue; one bad image must not end an overnight run.
            text, n_tokens, prompt_tokens = "", 0, 0
            fields, notes = {}, {}
            parse_error = f"GenerationError: {type(exc).__name__}: {exc}"
        elapsed = time.perf_counter() - began

        row = cm.make_prediction_row(
            image_key=key, cohort="MIDRC", subcohort="MIDRC",
            filename=key.split("::")[-1], held_out_fold=int(job["fold"]),
            agent="REASONER", arm=job.get("output_arm", job["name"]),
            view="v0_whole", task="reasoner_aggregation",
            covid_pred=fields.get("covid_pred"), covid_score=fields.get("covid_score"),
            mrale_total=fields.get("mrale_total"), mrale_right=fields.get("mrale_right"),
            mrale_left=fields.get("mrale_left"),
            extent_right=fields.get("extent_right"), density_right=fields.get("density_right"),
            extent_left=fields.get("extent_left"), density_left=fields.get("density_left"),
            gt_covid=truth.get("gt_covid"), gt_mrale_total=int(truth["gt_mrale_total"]),
            valid=parse_error is None, parse_error=parse_error,
            raw_output=text[:2000], seconds=round(elapsed, 3),
            model_id=job["spec"]["model_id"],
            model_revision=MODEL_REVISIONS.get(job["spec"]["model_id"]),
            extra={"reasoner": job["reasoner"], "setting": job["setting"],
                   "roster": job["roster"], "fold": job["fold"],
                   "actual_data_fold": int(truth["fold"]),
                   "score_role": job.get("score_role", "outer_test"),
                   "provenance": provenance, "notes": notes,
                   "n_completion_tokens": n_tokens, "n_prompt_tokens": prompt_tokens,
                   "fingerprint": fingerprint})
        cm.append_jsonl(path, row)

        if index % FLUSH_EVERY == 0 or index == len(pending):
            rate = index / max(time.perf_counter() - started, 1e-9)
            print(f"    [{index}/{len(pending)}] {rate:.2f} img/s, "
                  f"~{(len(pending) - index) / max(rate, 1e-9) / 60:.1f} min remaining")
    return path


def run_jobs(jobs, title):
    """Group by (reasoner, fold) so each model is loaded once."""
    print("=" * 78)
    print(title)
    print("=" * 78)
    groups = defaultdict(list)
    for job in jobs:
        groups[(job["reasoner"], job["fold"])].append(job)

    outputs = {}
    for (reasoner, fold), group in groups.items():
        pending_total = 0
        for job in group:
            forced = (FORCE_RECOMPUTE_ALL or job['name'] in FORCE_RECOMPUTE_CONFIGS
                      or journal_name(job) in FORCE_RECOMPUTE_CONFIGS)
            path = JOURNAL_DIR / f"{journal_name(job)}.jsonl"
            stamp = JOURNAL_DIR / f"{journal_name(job)}.fingerprint.json"
            fresh = (stamp.is_file() and json.loads(stamp.read_text(encoding="utf-8"))
                     .get("fingerprint") == config_fingerprint(job))
            done = scr.journal_keys(path) if fresh and not forced else set()
            pending_total += sum(1 for k in map(str, job["rows"].index) if k not in done)
        if pending_total == 0 and not FORCE_RECOMPUTE_ALL:
            print(f"[{reasoner} / fold {fold}] nothing pending; model not loaded")
            continue

        print(f"[{reasoner} / fold {fold}] {pending_total:,} images pending across "
              f"{len(group)} configuration(s)")
        scr.check_session_deadline(
            SESSION_DEADLINE, f"NB 15 before loading {reasoner} fold {fold}")
        try:
            model, processor, _ = load_reasoner(reasoner, group[0]["spec"], fold)
        except Exception as exc:
            raise RuntimeError(
                f"Required reasoner {reasoner} fold {fold} could not load; refusing to "
                "silently omit an E2 candidate. Remove it explicitly from "
                "REASONER_CANDIDATES or provide its adapter.") from exc
        try:
            for job in group:
                outputs[journal_name(job)] = run_job(job, model, processor)
        finally:
            release(model, processor)
        print(f"    variant used: {IMAGE_TEMPLATE_VARIANT.get(reasoner)}")
    return outputs


def read_config(name):
    """Concatenate a configuration's per-fold journals, newest state on disk."""
    rows = {}
    for path in scr.active_fold_journals(JOURNAL_DIR, name):
        for row in cm.read_jsonl(path):
            rows[(int(row.get("held_out_fold", -1)), str(row.get("image_key")))] = row
    return list(rows.values())


def evaluate_rows(rows, label):
    metrics = OrderedDict([("arm", label), ("n", len(rows))])
    scored = [r for r in rows if r.get("gt_mrale_total") is not None]
    if scored:
        m = cm.mrale_metrics(scored)
        metrics.update({"mae": round(m["mae"], 4), "rmse": round(m["rmse"], 4),
                        "qwk": round(m.get("qwk", float("nan")), 4),
                        "spearman": round(m.get("spearman_rho", float("nan")), 4),
                        "within1": round(m.get("within1_accuracy", float("nan")), 4),
                        "coverage": round(m.get("coverage", float("nan")), 4)})
    covid_rows = [r for r in rows if r.get("gt_covid") in {"Yes", "No"}]
    if covid_rows:
        c = cm.classification_metrics(
            [r["gt_covid"] for r in covid_rows], [r.get("covid_pred") for r in covid_rows],
            [r.get("covid_score") for r in covid_rows])
        score_coverage = float(np.mean([r.get("covid_score") is not None
                                        for r in covid_rows]))
        metrics.update({"covid_auroc": round(c.get("auroc", float("nan")), 4),
                        "covid_balanced_accuracy": round(
                            c.get("balanced_accuracy", float("nan")), 4),
                        "covid_score_coverage": round(score_coverage, 4),
                        "covid_score_decision_agreement": round(
                            c.get("score_decision_agreement", float("nan")), 4)})
    o = cm.localization_free_metrics(rows)
    metrics["valid_rate"] = round(o.get("output.valid_rate", float("nan")), 4)
    metrics["median_seconds"] = round(o.get("output.median_seconds", float("nan")), 2)
    return metrics


print("Run engine ready. Journals:", JOURNAL_DIR)

## 10. E2 — which model should be the reasoner

Five candidates on the full roster with a fixed E3 setting, scored on the evaluation half of
the selection split. Everything except the model identity is held constant, so the difference
is attributable to the model.

The LoRA candidates are included because a reasoner that has itself been trained on this task
is a different proposition from a general instruction-tuned model, and referee 1.2 asked
whether the framework needs a specialised reasoner at all. If a base model matches its
fine-tuned counterpart, that is the cheaper system and the honest recommendation.

In [ ]:
E2_SETTING = "E3b"     # reliability shown but not ranked: neutral ground for comparing models
e2_rows = []

if RUN_E2:
    jobs = [make_job(f"E2_{name}", name, FULL_ROSTER, E2_SETTING, SELECTION_FOLD,
                     SELECTION_EVAL, SELECTION_CONTEXT)
            for name in REASONER_CANDIDATES]
    run_jobs(jobs, f"E2 — reasoner identity ({len(jobs)} configurations, "
                   f"{len(SELECTION_EVAL):,} images each)")
    for name in REASONER_CANDIDATES:
        rows = read_config(f"E2_{name}")
        if rows:
            entry = {"reasoner": name, **evaluate_rows(rows, f"E2_{name}")}
            values = FRAME_BY_FOLD[SELECTION_FOLD]
            calls = COVID_PRED_BY_FOLD[SELECTION_FOLD]
            for agent in FULL_ROSTER:
                mrale_pairs = [(float(r["mrale_total"]), values.at[str(r["image_key"]), agent])
                               for r in rows if r.get("mrale_total") is not None
                               and str(r["image_key"]) in values.index
                               and agent in values.columns
                               and _finite(values.at[str(r["image_key"]), agent])]
                if mrale_pairs:
                    entry[f"agreement_mrale_{agent}"] = round(float(np.mean(
                        [round(a) == round(float(b)) for a, b in mrale_pairs])), 4)
                covid_pairs = [(r.get("covid_pred"), calls.at[str(r["image_key"]), agent])
                               for r in rows if str(r["image_key"]) in calls.index
                               and agent in calls.columns]
                covid_pairs = [(a, b) for a, b in covid_pairs
                               if a in {"Yes", "No"} and b in {"Yes", "No"}]
                if covid_pairs:
                    entry[f"agreement_covid_{agent}"] = round(float(np.mean(
                        [a == b for a, b in covid_pairs])), 4)
            e2_rows.append(entry)

e2 = pd.DataFrame(e2_rows)
if len(e2):
    e2 = e2.sort_values("mae")
    e2.to_csv(NB15_DIR / "e2_reasoner_metrics.csv", index=False)
    print()
    print(e2[["reasoner", "n", "mae", "qwk", "valid_rate", "covid_auroc",
              "median_seconds"]].to_string(index=False))
    usable = e2[e2["valid_rate"] >= 0.99]
    if not len(usable):
        print()
        print("No candidate reached a 0.99 strict-schema rate. The best available is used so the "
              "run can continue, but the gate in section 14 will fail and the arm cannot be "
              "reported.")
        usable = e2
    BEST_REASONER = usable.iloc[0]["reasoner"]
    margin = (float(usable.iloc[1]["mae"] - usable.iloc[0]["mae"]) if len(usable) > 1
              else float("nan"))
    print()
    print(f"E2 winner: {BEST_REASONER}"
          + (f" (beats the runner-up by {margin:.4f} MAE)" if math.isfinite(margin) else ""))
    if math.isfinite(margin) and margin < 0.10:
        print("  That margin is small enough to be noise. Report the candidates as equivalent")
        print("  and choose on cost, not on this ordering.")
else:
    BEST_REASONER = list(REASONER_CANDIDATES)[0]
    print(f"E2 did not run; defaulting to {BEST_REASONER}")

## 11. E3 — does telling the reasoner how reliable each agent is actually help?

The winning reasoner, the full roster, five nested settings, same evaluation rows. This is the
protocol's RQ7 and it has a genuinely uncertain answer: the Stage B results show the agents are
close enough in accuracy that reliability weighting may have little to work with.

A flat E3 ladder is a publishable finding — it says the reasoner cannot exploit reliability
information at this level of agent disagreement — provided it is reported as such rather than
buried.

In [ ]:
e3_rows = []
if RUN_E3:
    jobs = [make_job(f"E3_{setting}", BEST_REASONER, FULL_ROSTER, setting, SELECTION_FOLD,
                     SELECTION_EVAL, SELECTION_CONTEXT)
            for setting in E3_SETTINGS]
    run_jobs(jobs, f"E3 — metrics awareness with {BEST_REASONER} "
                   f"({len(jobs)} settings, {len(SELECTION_EVAL):,} images each)")
    for setting in E3_SETTINGS:
        rows = read_config(f"E3_{setting}")
        if rows:
            entry = {"setting": setting, **evaluate_rows(rows, f"E3_{setting}")}
            # Does the answer actually move toward the more reliable agents?
            best_agent = min(
                (a for a in FULL_ROSTER
                 if _finite(SELECTION_CONTEXT["reliability"].get(a, {}).get("mrale_mae"))),
                key=lambda a: SELECTION_CONTEXT["reliability"][a]["mrale_mae"], default=None)
            worst_agent = max(
                (a for a in FULL_ROSTER
                 if _finite(SELECTION_CONTEXT["reliability"].get(a, {}).get("mrale_mae"))),
                key=lambda a: SELECTION_CONTEXT["reliability"][a]["mrale_mae"], default=None)
            if best_agent and worst_agent and best_agent != worst_agent:
                deltas = []
                for row in rows:
                    key = str(row["image_key"])
                    selection_values = FRAME_BY_FOLD[SELECTION_FOLD]
                    if row.get("mrale_total") is None or key not in selection_values.index:
                        continue
                    prediction = float(row["mrale_total"])
                    best_value = selection_values.at[key, best_agent]
                    worst_value = selection_values.at[key, worst_agent]
                    if _finite(best_value) and _finite(worst_value):
                        deltas.append(abs(prediction - float(best_value))
                                      - abs(prediction - float(worst_value)))
                if deltas:
                    entry["pull_toward_reliable"] = round(-float(np.mean(deltas)), 4)
            e3_rows.append(entry)

e3 = pd.DataFrame(e3_rows)
if len(e3):
    e3.to_csv(NB15_DIR / "e3_metrics_aware.csv", index=False)
    columns = [c for c in ["setting", "n", "mae", "qwk", "valid_rate", "covid_auroc",
                           "pull_toward_reliable"] if c in e3.columns]
    print()
    print(e3[columns].to_string(index=False))
    print()
    print("`pull_toward_reliable` > 0 means the reasoner's answers sit closer to the most")
    print("reliable agent than to the least. If E3d and E3e do not raise it above E3a, the")
    print("reliability instruction is being read but not acted on — report that.")

    ranked = e3.sort_values("mae")
    BEST_SETTING = ranked.iloc[0]["setting"]
    spread = float(e3["mae"].max() - e3["mae"].min())
    print()
    print(f"E3 winner: {BEST_SETTING}   (ladder spans {spread:.4f} MAE)")
    if spread < 0.10:
        print("  The whole ladder fits inside the noise band. The honest claim is that")
        print("  metrics awareness does not measurably change this system, not that E3"
              f"{BEST_SETTING[-1]} is best.")
else:
    BEST_SETTING = "E3b"
    print(f"E3 did not run; defaulting to {BEST_SETTING}")

## 12. E1 — the roster ablation, on held-out folds

The winning reasoner and setting, twelve rosters, all five folds. Everything above this point
was selection; this is the evaluation.

Two families of arm:

- **Add-one-in** (E1-0 through E1-full) traces what each agent adds as the roster grows. E1-0
  is the reasoner with no agents at all, which is the honest floor: if the full roster does not
  beat it, the agents are decorative.
- **Leave-one-out** (E1-L_minus_*) removes one agent from the full roster. This is what
  referee 1.2 asked for, and the arm that licenses deleting a module.

All arms are scored on the same images, so every delta is paired.

In [ ]:
e1_predictions, e1_rows, per_fold_metrics = {}, [], defaultdict(dict)

if RUN_E1:
    jobs = []
    for fold in E1_FOLDS:
        test_rows = frame[frame["fold"] == fold]
        if E1_MAX_IMAGES_PER_FOLD and len(test_rows) > E1_MAX_IMAGES_PER_FOLD:
            test_rows = test_rows.sample(n=E1_MAX_IMAGES_PER_FOLD,
                                         random_state=SEED + fold).sort_index()
        for name, roster in E1_ROSTERS.items():
            jobs.append(make_job(name, BEST_REASONER, roster, BEST_SETTING, fold,
                                 test_rows, FOLD_CONTEXT[fold]))
    total = sum(len(job["rows"]) for job in jobs)
    run_jobs(jobs, f"E1 — roster ablation with {BEST_REASONER} / {BEST_SETTING} "
                   f"({len(E1_ROSTERS)} rosters x {len(E1_FOLDS)} folds = {total:,} "
                   "generations at most)")

    for name in E1_ROSTERS:
        rows = read_config(name)
        if not rows:
            continue
        e1_predictions[name] = {str(r["image_key"]): r for r in rows}
        entry = {"arm": name, "also_known_as": ",".join(E1_ALIASES.get(name, [])),
                 "roster": ",".join(E1_ROSTERS[name]) or "(none)",
                 "n_agents": len(E1_ROSTERS[name])}
        entry.update({k: v for k, v in evaluate_rows(rows, name).items() if k != "arm"})
        # Per-fold metrics feed the cross-validation confidence interval. The protocol
        # reports mean +/- t-based 95% CI over folds, matching every Stage B table.
        for fold in E1_FOLDS:
            fold_rows = [r for r in rows if r.get("held_out_fold") == fold]
            if not fold_rows:
                continue
            fold_metrics = dict(cm.mrale_metrics(fold_rows))
            labelled = [r for r in fold_rows if r.get("gt_covid") in {"Yes", "No"}]
            if labelled:
                covid = cm.classification_metrics(
                    [r["gt_covid"] for r in labelled],
                    [r.get("covid_pred") for r in labelled],
                    [r.get("covid_score") for r in labelled])
                fold_metrics.update({f"covid.{k}": v for k, v in covid.items()})
            per_fold_metrics[name][fold] = fold_metrics
        if per_fold_metrics[name]:
            # aggregate_over_folds takes {fold -> metrics} and returns a list of rows.
            aggregate = {row["metric"]: row
                         for row in cm.aggregate_over_folds(per_fold_metrics[name])}
            for metric in ["mae", "qwk", "covid.auroc"]:
                row = aggregate.get(metric)
                if row:
                    entry[f"{metric}_cv_mean"] = round(row["mean"], 4)
                    entry[f"{metric}_cv_ci95"] = (
                        None if not math.isfinite(row["ci95_margin"])
                        else round(row["ci95_margin"], 4))
        e1_rows.append(entry)
        cm.write_jsonl(NB15_DIR / f"reasoner_predictions_{name}.jsonl", rows)

# NB 18 needs fold-owner scores on each fold's inner-validation membership to select
# Youden-J and fixed-sensitivity thresholds without looking at outer-test outcomes. Use a
# distinct journal namespace so completed E1 outer-test work remains untouched. Every
# journal is append-only and keyed by image, so a 36-hour interruption resumes at the
# first unfinished image rather than repeating completed generations.
INNER_THRESHOLD_JOB = "NB18-inner-E1-full"
inner_threshold_status = []
if RUN_INNER_THRESHOLD_SCORES:
    inner_jobs = []
    expected_inner_by_fold = {}
    for fold in range(N_FOLDS):
        fold_frame = FRAME_BY_FOLD[fold]
        inner_rows = fold_frame[(fold_frame[f"inner_fold_{fold}"] == 1)
                                & (fold_frame["fold"] != fold)].sort_index()
        expected_inner_by_fold[fold] = set(map(str, inner_rows.index))
        missing_paths = sorted(key for key in expected_inner_by_fold[fold]
                               if not IMAGE_PATHS.get(key))
        if missing_paths:
            raise FileNotFoundError(
                f"Fold {fold}: {len(missing_paths)} inner-validation images lack a V0/V1 "
                f"path, e.g. {missing_paths[:3]}. Re-run/check NB 04.")
        inner_jobs.append(make_job(
            INNER_THRESHOLD_JOB, BEST_REASONER, FULL_ROSTER, BEST_SETTING, fold,
            inner_rows, FOLD_CONTEXT[fold], output_arm=FULL_ROSTER_ARM,
            score_role="inner_validation_threshold_selection"))
    run_jobs(inner_jobs,
             f"NB 18 threshold scores — {FULL_ROSTER_ARM} with {BEST_REASONER} / "
             f"{BEST_SETTING} ({sum(len(job['rows']) for job in inner_jobs):,} "
             "generations at most; append-only resume)")

    all_inner_rows = read_config(INNER_THRESHOLD_JOB)
    for fold in range(N_FOLDS):
        expected = expected_inner_by_fold[fold]
        rows = [row for row in all_inner_rows
                if int(row.get("held_out_fold", -1)) == fold
                and str(row.get("image_key")) in expected
                and str(row.get("arm")) == FULL_ROSTER_ARM]
        by_key = {str(row["image_key"]): row for row in rows}
        missing = sorted(expected - set(by_key))
        if missing:
            raise RuntimeError(
                f"Fold {fold}: {len(missing)} inner reasoner scores remain missing, e.g. "
                f"{missing[:3]}. Re-run this cell; completed journal rows will be reused.")
        ordered = [by_key[key] for key in sorted(expected)]
        fold_dir = NB15_DIR / "folds" / f"fold_{fold}"
        fold_dir.mkdir(parents=True, exist_ok=True)
        output_path = fold_dir / "inner_validation_score_records.jsonl"
        cm.write_jsonl(output_path, ordered)
        labels = {str(row.get("gt_covid")) for row in ordered}
        score_coverage = (float(np.mean([row.get("covid_score") is not None
                                         for row in ordered])) if ordered else 0.0)
        inner_threshold_status.append({
            "fold": fold, "n": len(ordered), "labels": sorted(labels),
            "score_coverage": score_coverage, "path": str(output_path)})
        print(f"  fold {fold}: wrote {len(ordered):,} inner reasoner scores "
              f"(coverage {score_coverage:.1%})")

e1 = pd.DataFrame(e1_rows)
if len(e1):
    e1.to_csv(NB15_DIR / "e1_roster_metrics.csv", index=False)
    columns = [c for c in ["arm", "also_known_as", "n_agents", "n", "mae",
                           "mae_cv_mean", "mae_cv_ci95", "qwk", "valid_rate",
                           "covid_auroc"] if c in e1.columns]
    print()
    print(e1[columns].to_string(index=False))
    cm.write_json(NB15_DIR / "per_fold_metrics.json", cm.json_safe(dict(per_fold_metrics)))

## 13. Paired contrasts — leave-one-out deltas, and E7f against E7d

A leave-one-out arm and the full roster answer the *same* images, so the comparison is paired
and a cluster bootstrap over patient groups gives an honest interval. An unpaired comparison of
two MAEs, each with its own confidence interval, would be far less sensitive and would let a
real effect hide inside overlapping bars.

The bootstrap resamples **patient groups**, not images. Two radiographs of the same patient are
not independent, and resampling images would produce intervals that are too narrow.

`e7f_vs_e7d.json` is the file the manuscript's central claim rests on.

In [ ]:
GROUP_OF = {str(k): str(v) for k, v in frame["group_id"].astype(str).items()}


def paired_bootstrap(keys, errors_a, errors_b, n_bootstrap=N_BOOTSTRAP, seed=SEED):
    """Cluster bootstrap of mean(errors_a) - mean(errors_b) over patient groups."""
    by_group = defaultdict(list)
    for key, a, b in zip(keys, errors_a, errors_b):
        by_group[GROUP_OF.get(key, key)].append((a, b))
    groups = list(by_group)
    if len(groups) < 5:
        return {"delta": float(np.mean(errors_a) - np.mean(errors_b)),
                "ci_low": float("nan"), "ci_high": float("nan"), "n_groups": len(groups)}
    rng = np.random.default_rng(seed)
    observed = float(np.mean(errors_a) - np.mean(errors_b))
    draws = np.empty(n_bootstrap)
    index = np.arange(len(groups))
    for i in range(n_bootstrap):
        picked = rng.choice(index, size=len(groups), replace=True)
        pairs = [pair for j in picked for pair in by_group[groups[j]]]
        a = np.fromiter((p[0] for p in pairs), dtype=float, count=len(pairs))
        b = np.fromiter((p[1] for p in pairs), dtype=float, count=len(pairs))
        draws[i] = a.mean() - b.mean()
    low, high = np.percentile(draws, [2.5, 97.5])
    return {"delta": observed, "ci_low": float(low), "ci_high": float(high),
            "n_groups": len(groups), "n_pairs": len(keys),
            "excludes_zero": bool(low > 0 or high < 0)}


def penalised_error(row):
    if row.get("mrale_total") is None:
        return cm.INVALID_TOTAL_PENALTY
    return abs(float(row["mrale_total"]) - float(row["gt_mrale_total"]))


def paired_errors(arm_a, arm_b):
    a_map, b_map = e1_predictions.get(arm_a, {}), e1_predictions.get(arm_b, {})
    keys = sorted(set(a_map) & set(b_map))
    return keys, [penalised_error(a_map[k]) for k in keys], \
        [penalised_error(b_map[k]) for k in keys]


contrast_rows = []
if FULL_ROSTER_ARM in e1_predictions:
    for name in E1_ROSTERS:
        if name == FULL_ROSTER_ARM or name not in e1_predictions:
            continue
        keys, errors_arm, errors_full = paired_errors(name, FULL_ROSTER_ARM)
        if len(keys) < 30:
            continue
        result = paired_bootstrap(keys, errors_arm, errors_full)
        def covid_error(row):
            truth = row.get("gt_covid")
            prediction = row.get("covid_pred")
            return float(prediction != truth) if truth in {"Yes", "No"} else float("nan")
        covid_arm = [covid_error(e1_predictions[name][k]) for k in keys]
        covid_full = [covid_error(e1_predictions[FULL_ROSTER_ARM][k]) for k in keys]
        covid_keep = [i for i, value in enumerate(covid_arm) if math.isfinite(value)
                      and math.isfinite(covid_full[i])]
        covid_result = paired_bootstrap(
            [keys[i] for i in covid_keep], [covid_arm[i] for i in covid_keep],
            [covid_full[i] for i in covid_keep]) if len(covid_keep) >= 5 else {}
        contrast_rows.append({
            "arm": name, "vs": FULL_ROSTER_ARM, "n_paired": len(keys),
            "delta_mae": round(result["delta"], 4),
            "ci_low": round(result["ci_low"], 4), "ci_high": round(result["ci_high"], 4),
            "significant": result.get("excludes_zero", False),
            "delta_covid_error": (round(covid_result["delta"], 4)
                                     if covid_result else None),
            "covid_ci_low": (round(covid_result["ci_low"], 4)
                                if covid_result else None),
            "covid_ci_high": (round(covid_result["ci_high"], 4)
                                 if covid_result else None),
            "reading": ("removing this agent HURTS" if result["delta"] > 0 else
                        "removing this agent HELPS" if result["delta"] < 0 else "no change")})

contrasts = pd.DataFrame(contrast_rows)
if len(contrasts):
    contrasts.to_csv(NB15_DIR / "e1_paired_contrasts.csv", index=False)
    print(contrasts.to_string(index=False))
    print()
    dispensable = contrasts[(contrasts["arm"].str.contains("minus"))
                            & (~contrasts["significant"])]
    if len(dispensable):
        names = [a.split("minus_")[-1] for a in dispensable["arm"]]
        print(f"INCONCLUSIVE CONTRIBUTION: {names}")
        print("  A confidence interval that straddles zero is absence of evidence, not")
        print("  evidence of equivalence. Mark these agents for a pre-specified equivalence")
        print("  margin analysis in NB 17 before deleting them.")
    else:
        print("Every agent's removal moves the paired interval away from zero: the roster is "
              "minimal.")

# ---- E7f against NB 14's best stacking baseline, on the same images ------------------------
e7f_result = {}
fusion_path = NB14_DIR / "fusion_predictions.jsonl"
if FULL_ROSTER_ARM in e1_predictions and fusion_path.is_file() and e7f_target:
    target_arm = e7f_target.get("best_stacking_arm")
    fusion_rows_all = cm.read_jsonl(fusion_path)
    fusion = {str(r["image_key"]): r for r in fusion_rows_all
              if r.get("arm") == target_arm
              and r.get("task") == "mrale_prediction"}
    reasoner = e1_predictions[FULL_ROSTER_ARM]
    keys = sorted(set(fusion) & set(reasoner))
    if len(keys) >= 30:
        errors_reasoner = [penalised_error(reasoner[k]) for k in keys]
        errors_fusion = [penalised_error(fusion[k]) for k in keys]
        result = paired_bootstrap(keys, errors_reasoner, errors_fusion)
        e7f_result = {
            "e7f_arm": FULL_ROSTER_ARM, "baseline_arm": target_arm,
            "n_paired_images": len(keys), "n_groups": result["n_groups"],
            "e7f_mae": round(float(np.mean(errors_reasoner)), 4),
            "baseline_mae": round(float(np.mean(errors_fusion)), 4),
            "delta_mae": round(result["delta"], 4),
            "ci_low": round(result["ci_low"], 4), "ci_high": round(result["ci_high"], 4),
            "reasoner_wins": bool(result["delta"] < 0 and result.get("excludes_zero")),
        }
        covid_target = e7f_target.get("best_covid_stacking_arm")
        covid_fusion = {str(r["image_key"]): r for r in fusion_rows_all
                        if r.get("arm") == covid_target
                        and r.get("task") == "covid_classification"}
        covid_keys = sorted(set(covid_fusion) & set(reasoner))
        if len(covid_keys) >= 30:
            truth = [reasoner[k]["gt_covid"] for k in covid_keys]
            reasoner_metrics = cm.classification_metrics(
                truth, [reasoner[k].get("covid_pred") for k in covid_keys],
                [reasoner[k].get("covid_score") for k in covid_keys])
            fusion_metrics = cm.classification_metrics(
                truth, [covid_fusion[k].get("covid_pred") for k in covid_keys],
                [covid_fusion[k].get("covid_score") for k in covid_keys])
            e7f_result.update({
                "covid_baseline_arm": covid_target, "n_covid_paired_images": len(covid_keys),
                "e7f_covid_auroc": round(reasoner_metrics.get("auroc", float("nan")), 4),
                "baseline_covid_auroc": round(fusion_metrics.get("auroc", float("nan")), 4),
                "covid_delong_pending_nb17": True})
        e7f_result["verdict"] = (
            "The reasoner beats learned stacking on the same images, with a paired interval "
            "excluding zero. An accuracy claim is supported."
            if e7f_result["reasoner_wins"] else
            "The reasoner does NOT beat learned stacking by more than noise. Protocol risk K1 "
            "applies: report the contribution as auditability and per-case evidence, not "
            "accuracy, and state this comparison explicitly rather than omitting it.")
        cm.write_json(NB15_DIR / "e7f_vs_e7d.json", e7f_result)
        print()
        print("E7f vs E7d (paired, same images, cluster bootstrap over patients):")
        print(f"  E7f {e7f_result['e7f_mae']:.4f} vs {target_arm} "
              f"{e7f_result['baseline_mae']:.4f}   delta "
              f"{e7f_result['delta_mae']:+.4f} [{e7f_result['ci_low']:+.4f}, "
              f"{e7f_result['ci_high']:+.4f}]")
        print(f"  {e7f_result['verdict']}")
else:
    print("E7f comparison skipped: needs both NB 14's fusion predictions and an E1-full run.")

## 14. Reasoning traces and the provenance audit

The protocol's gate says *agent-output provenance recoverable for every final answer*. That is
checked by **recomputation**, not by the presence of a field: for each answer, the agent values
recorded in its provenance block are compared against the registry. If they disagree, the
prompt did not show what the trace claims it showed, and every auditability claim in the paper
is void.

Two further counts come out of the same pass:

- **Fabricated citations** — the model naming an agent it was never given. A framework that
  cites non-existent evidence is worse than one that cites none.
- **Override rate** — how often the final answer falls outside the range the agents spanned.
  A reasoner that never leaves that range is doing arithmetic, and the paper should not
  describe it as reading the image.

In [ ]:
trace_rows, provenance_failures, fabricated, override_count, in_range = [], [], [], 0, 0

audit_arm = FULL_ROSTER_ARM if FULL_ROSTER_ARM in e1_predictions else (
    next(iter(e1_predictions)) if e1_predictions else None)

if audit_arm:
    for key, row in sorted(e1_predictions[audit_arm].items()):
        extra = row.get("extra") or {}
        provenance = extra.get("provenance") or {}
        shown = provenance.get("agents") or {}
        notes = extra.get("notes") or {}

        # 1. Recompute: does the block match the registry for this image?
        for agent, entry in shown.items():
            fold = int(row.get("held_out_fold"))
            values_frame = FRAME_BY_FOLD[fold]
            if key not in values_frame.index or agent not in values_frame.columns:
                provenance_failures.append(f"{key}: {agent} not in the registry frame")
                continue
            if not _finite(entry.get("mrale_total")):
                continue
            registry_value = values_frame.at[key, agent]
            if not _finite(registry_value):
                provenance_failures.append(f"{key}: prompt recorded numeric {agent} but registry has none")
                continue
            registry_value = float(registry_value)
            if abs(float(entry["mrale_total"]) - registry_value) > 1e-6:
                provenance_failures.append(
                    f"{key}: prompt showed {agent}={entry['mrale_total']} but the registry "
                    f"holds {registry_value}")
        if not shown and extra.get("roster"):
            provenance_failures.append(f"{key}: roster {extra['roster']} but no evidence "
                                       "recorded")

        # 2. Fabricated citations
        if notes.get("agents_fabricated"):
            fabricated.append((key, notes["agents_fabricated"]))

        # 3. Override: is the answer outside the agents' span?
        values = [v["mrale_total"] for v in shown.values()
                  if _finite(v.get("mrale_total"))]
        prediction = row.get("mrale_total")
        if values and prediction is not None:
            if min(values) <= float(prediction) <= max(values):
                in_range += 1
            else:
                override_count += 1

        trace_rows.append({
            "image_key": key, "arm": audit_arm, "fold": row.get("held_out_fold"),
            "reasoner": extra.get("reasoner"), "setting": extra.get("setting"),
            "agents_shown": {a: v.get("mrale_total") for a, v in shown.items()},
            "findings_shown": {a: v.get("findings") for a, v in shown.items()
                                if v.get("findings")},
            "reliability_shown": {a: v.get("shown_mae") for a, v in shown.items()
                                  if v.get("shown_mae") is not None},
            "conflict_flagged": provenance.get("conflict_flagged"),
            "agent_spread": provenance.get("agent_spread"),
            "prompt_sha256": provenance.get("prompt_sha256"),
            "final_mrale_total": prediction,
            "final_covid": row.get("covid_pred"),
            "ground_truth_mrale": row.get("gt_mrale_total"),
            "ground_truth_covid": row.get("gt_covid"),
            "agents_cited": notes.get("agents_cited"),
            "rationale": notes.get("rationale"),
            "valid": row.get("valid"), "parse_error": row.get("parse_error"),
        })

    cm.write_jsonl(NB15_DIR / "reasoning_traces.jsonl", trace_rows)
    total_ranged = in_range + override_count
    print(f"Traces written for {len(trace_rows):,} cases of {audit_arm}")
    print(f"  provenance mismatches : {len(provenance_failures):,}")
    print(f"  fabricated citations  : {len(fabricated):,}")
    if total_ranged:
        print(f"  answers outside the agents' span: {override_count:,}/{total_ranged:,} "
              f"({override_count / total_ranged:.1%})")
        if override_count / total_ranged < 0.02:
            print("    The reasoner almost never leaves the range its agents span, so on this")
            print("    evidence it is aggregating rather than re-reading the image. Say so.")
    if fabricated[:3]:
        print("  examples of fabricated citations:")
        for key, names in fabricated[:3]:
            print(f"    {key}: cited {names}")
else:
    print("No E1 predictions to audit.")

## 15. Run configuration and gate

Two blocking conditions, both from the protocol:

1. **Strict complete-schema rate >= 0.99 for every final E1 roster configuration.** A merely
   parseable object is not enough: all component, regional, total, formula and COVID fields
   must be valid. E2 reasoner candidates and E3 prompt settings are screening experiments:
   candidates below 0.99 are quarantined in `usability.json` and retained as negative
   output-integrity results, rather than aborting an otherwise valid final E1 run. Penalised
   metrics still include every attempted image under protocol 7.4.
2. **Provenance recoverable for every final answer.** This is the auditability claim, and it is
   the claim most likely to survive if the accuracy claim does not.

Everything else is a warning that shapes what the manuscript may say — including the E7f
verdict, which is a warning rather than a failure precisely because a negative result is a
legitimate outcome that must be reported, not a reason to stop.

In [ ]:
failures, warnings = [], []

# ---- Gate 1: strict complete-schema rate ---------------------------------------------------
# E1 is the final held-out roster ablation and is blocking. E2/E3 are selection/screening
# experiments: a candidate that cannot follow the schema is quarantined as an empirical
# negative result, not allowed to invalidate final E1 configurations that did pass.
SCHEMA_RATE_THRESHOLD = 0.99
valid_rates, configuration_sizes = {}, {}
for table, key in [(e2, "reasoner"), (e3, "setting"), (e1, "arm")]:
    if len(table) and "valid_rate" in table.columns:
        for row in table.to_dict("records"):
            name = str(row[key])
            valid_rates[name] = row["valid_rate"]
            n_value = row.get("n", 0)
            configuration_sizes[name] = int(n_value) if pd.notna(n_value) else 0

primary_names = set(map(str, e1["arm"])) if len(e1) and "arm" in e1.columns else set()
low = {k: float(v) for k, v in valid_rates.items()
       if pd.notna(v) and float(v) < SCHEMA_RATE_THRESHOLD}
low_primary = {k: v for k, v in low.items() if k in primary_names}
low_screening = {k: v for k, v in low.items() if k not in primary_names}

def schema_detail(entries):
    parts = []
    for name, rate in sorted(entries.items(), key=lambda item: item[1]):
        n = configuration_sizes.get(name, 0)
        approximate_invalid = max(0, n - int(round(rate * n))) if n else None
        count = f"; ~{approximate_invalid}/{n} invalid" if n else ""
        parts.append(f"{name} {rate:.4f}{count}")
    return ", ".join(parts)

if low_primary:
    failures.append(
        f"{len(low_primary)} final E1 configuration(s) below the "
        f"{SCHEMA_RATE_THRESHOLD:.2f} strict-schema rate: {schema_detail(low_primary)}. "
        f"Inspect `raw_output` in the corresponding journal under {JOURNAL_DIR.name}/.")
elif primary_names:
    print(f"Strict-schema rate >= {SCHEMA_RATE_THRESHOLD:.2f} for all "
          f"{len(primary_names)} final E1 configurations")

if low_screening:
    warnings.append(
        f"{len(low_screening)} E2/E3 screening configuration(s) below the "
        f"{SCHEMA_RATE_THRESHOLD:.2f} strict-schema target and quarantined from endpoint "
        f"reporting: {schema_detail(low_screening)}. Their output-integrity rates and "
        "protocol-7.4 penalised metrics remain valid negative results. Inspect "
        f"`raw_output` in the corresponding journal under {JOURNAL_DIR.name}/ before "
        "changing prompts. NV-Reason's native <think>/<answer> response is not JSON and "
        "must not be repaired into a strict-schema success.")

# ---- Gate 1b: protocol 7.2 token-score coverage and decision agreement ----------------
score_health = {}
for table, key in [(e2, "reasoner"), (e3, "setting"), (e1, "arm")]:
    if len(table):
        for row in table.to_dict("records"):
            score_health[str(row[key])] = {
                "coverage": row.get("covid_score_coverage"),
                "agreement": row.get("covid_score_decision_agreement")}
bad_scores = {name: health for name, health in score_health.items()
              if not _finite(health.get("coverage"))
              or health["coverage"] < 0.99
              or not _finite(health.get("agreement"))
              or health["agreement"] < 0.995}
bad_primary_scores = {k: v for k, v in bad_scores.items() if k in primary_names}
bad_screening_scores = {k: v for k, v in bad_scores.items() if k not in primary_names}
if bad_primary_scores:
    failures.append(
        f"Protocol 7.2 token scores fail coverage/agreement for "
        f"{len(bad_primary_scores)} final E1 configuration(s): "
        f"{dict(list(bad_primary_scores.items())[:5])}. AUROC/DeLong results "
        "must not be reported until this gate passes.")
if bad_screening_scores:
    warnings.append(
        f"Protocol 7.2 token scores are unusable for {len(bad_screening_scores)} E2/E3 "
        f"screening configuration(s): {dict(list(bad_screening_scores.items())[:5])}. "
        "Those configurations are excluded from AUROC/DeLong reporting.")

# ---- Gate 1c: NB 18 owner-fold threshold-score contract -------------------------------
if RUN_INNER_THRESHOLD_SCORES:
    bad_inner = [entry for entry in inner_threshold_status
                 if entry["n"] == 0
                 or not {"Yes", "No"}.issubset(set(entry["labels"]))
                 or entry["score_coverage"] < 0.99]
    if bad_inner:
        failures.append(
            f"NB 18 owner-fold inner reasoner scores are incomplete or single-class: "
            f"{bad_inner}. Re-run section 12; its append-only journals resume.")
    elif len(inner_threshold_status) != N_FOLDS:
        failures.append(
            f"NB 18 inner-score contract covers {len(inner_threshold_status)} folds, "
            f"expected {N_FOLDS}.")
    else:
        print(f"NB 18 INNER-SCORE CONTRACT PASSED: {FULL_ROSTER_ARM}, all folds.")

# ---- Gate 2: provenance ------------------------------------------------------------------
if audit_arm and provenance_failures:
    failures.append(
        f"{len(provenance_failures)} answers whose recorded evidence does not match the "
        f"registry, e.g. {provenance_failures[0]}. The auditability claim cannot be made.")
elif audit_arm:
    print(f"Provenance recomputed and matched for all {len(trace_rows):,} answers in "
          f"{audit_arm}")

if fabricated:
    warnings.append(
        f"{len(fabricated)} answers cite an agent that was not in their roster (e.g. "
        f"{fabricated[0][1]}). Citations are not reliable provenance for those cases.")

# ---- Warnings that shape the manuscript ---------------------------------------------------
if e7f_result:
    warnings.append(f"E7f vs {e7f_result['baseline_arm']}: delta "
                    f"{e7f_result['delta_mae']:+.4f} "
                    f"[{e7f_result['ci_low']:+.4f}, {e7f_result['ci_high']:+.4f}]. "
                    + e7f_result["verdict"])
elif RUN_E1:
    warnings.append("E7f was not compared against NB 14's stacking baseline, so the "
                    "framework's central accuracy claim is untested. Run NB 14, then re-run "
                    "section 13.")

if len(e1) and "E1-0_reasoner_only" in set(e1["arm"]) \
        and FULL_ROSTER_ARM in set(e1["arm"]):
    floor = float(e1.loc[e1["arm"] == "E1-0_reasoner_only", "mae"].iloc[0])
    full = float(e1.loc[e1["arm"] == FULL_ROSTER_ARM, "mae"].iloc[0])
    warnings.append(
        f"Reasoner alone {floor:.4f} MAE vs full roster {full:.4f} (delta {full - floor:+.4f}). "
        + ("The agents earn their place." if full < floor - 0.10 else
           "The agent roster does not measurably beat the reasoner working alone, which bounds "
           "what the framework can claim."))

if len(e3) and "mae" in e3.columns:
    spread = float(e3["mae"].max() - e3["mae"].min())
    if spread < 0.10:
        warnings.append(
            f"The E3 ladder spans only {spread:.4f} MAE. Metrics awareness does not measurably "
            "change this system; report the null rather than the winner.")

if len(e2) and "mae" in e2.columns and len(e2) > 1:
    margin = float(e2["mae"].iloc[1] - e2["mae"].iloc[0])
    if margin < 0.10:
        warnings.append(
            f"E2's top two reasoners differ by {margin:.4f} MAE, inside the noise band. "
            "Choose on cost and report them as equivalent.")

if len(contrasts):
    dispensable = contrasts[(contrasts["arm"].str.contains("minus"))
                            & (~contrasts["significant"])]
    if len(dispensable):
        warnings.append(
            "Agents with inconclusive removal effects (paired interval straddles zero): "
            + ", ".join(a.split("minus_")[-1] for a in dispensable["arm"])
            + ". Do not call them non-contributory without an equivalence-margin test.")

if absent:
    warnings.append(f"Agents {absent} were absent, so the E1 grid ablates a partial roster"
                    + (" and E1-L6 could not run." if "A6" in absent else "."))

no_placeholder = {k: v for k, v in IMAGE_TEMPLATE_VARIANT.items()
                  if isinstance(v, str) and v.startswith("NO_PLACEHOLDER")}
if no_placeholder:
    failures.append(
        f"No image placeholder could be rendered for {sorted(no_placeholder)}. The model never "
        "saw the radiograph; every answer from it is text-only and must not be reported. This "
        "is the failure that produced a 20-hour run at valid_rate 0.000 in NB 07.")

# ---- Usability, stamped onto the outputs --------------------------------------------------
usability = {}
for name, rate in valid_rates.items():
    usable = bool(pd.notna(rate) and rate >= 0.99)
    health = score_health.get(name, {})
    score_usable = bool(usable and _finite(health.get("coverage"))
                        and health["coverage"] >= 0.99
                        and _finite(health.get("agreement"))
                        and health["agreement"] >= 0.995)
    usability[name] = {"mrale_usable": usable, "score_usable": score_usable,
                       "valid_rate": (None if pd.isna(rate) else float(rate)),
                       "report_in_table": usable}
cm.write_json(NB15_DIR / "usability.json", usability)
for table, path in [(e1, "e1_roster_metrics.csv"), (e2, "e2_reasoner_metrics.csv"),
                    (e3, "e3_metrics_aware.csv")]:
    if len(table):
        key = "arm" if "arm" in table.columns else table.columns[0]
        table = table.copy()
        table["report_in_table"] = table[key].map(
            lambda v: usability.get(str(v), {}).get("report_in_table", False))
        table.to_csv(NB15_DIR / path, index=False)

cm.write_json(NB15_DIR / "run_config.json", {
    "written_utc": datetime.now(timezone.utc).isoformat(),
    "notebook": "15_llm_reasoner_aggregation.ipynb",
    "protocol_experiments": ["E1", "E2", "E3", "E7f"],
    "seed": SEED,
    "reasoner_selected": BEST_REASONER,
    "reasoner_spec": REASONER_CANDIDATES[BEST_REASONER],
    "reasoner_model_id": REASONER_CANDIDATES[BEST_REASONER]["model_id"],
    "setting_selected": BEST_SETTING,
    "roster_full": FULL_ROSTER,
    "full_roster_arm": FULL_ROSTER_ARM,
    "e1_aliases": E1_ALIASES,
    "agents_absent": absent,
    "n_e1_configurations": len(E1_ROSTERS),
    "e1_folds": E1_FOLDS,
    "selection_fold": SELECTION_FOLD,
    "selection_fit_images": len(SELECTION_FIT_KEYS),
    "selection_eval_images": int(len(SELECTION_EVAL)),
    "selection_max_images": SELECTION_MAX_IMAGES,
    "framework_denominator": int(len(frame)),
    "inner_threshold_scores_enabled": RUN_INNER_THRESHOLD_SCORES,
    "inner_threshold_score_status": inner_threshold_status,
    "inner_threshold_score_journal": INNER_THRESHOLD_JOB,
    "inner_threshold_score_role": "operating-point selection only; never outer-test evaluation",
    "max_new_tokens": MAX_NEW_TOKENS,
    "strict_schema_rate_threshold": SCHEMA_RATE_THRESHOLD,
    "strict_schema_gate_scope": "final_E1_configurations",
    "strict_schema_screening_quarantine": sorted(low_screening),
    "decoding": "greedy",
    "image_template_variant": IMAGE_TEMPLATE_VARIANT,
    "e7f_vs_e7d": e7f_result,
    "leakage_note": (
        "Reliability shown in E3 prompts for fold k comes only from rows NB 13 flagged "
        "inner_fold_k, which are drawn from the other four folds. E2 and E3 were selected on a "
        "group-disjoint half of fold "
        f"{SELECTION_FOLD}'s inner-validation pool, with reliability computed on the other "
        "half, so no selection decision used its own evaluation rows."),
})


def report(title, messages):
    print(title)
    for message in messages or []:
        print("  -", message)
    if not messages:
        print("  none")


print()
report("WARNINGS", warnings)
print()
report("FAILURES", failures)
cm.write_json(NB15_DIR / "gate_nb15.json",
              {"passed": not failures, "failures": failures, "warnings": warnings,
               "strict_schema_gate_scope": "final_E1_configurations",
               "strict_schema_rate_threshold": SCHEMA_RATE_THRESHOLD,
               "screening_configurations_quarantined": sorted(low_screening),
               "valid_rates": {k: (None if pd.isna(v) else float(v))
                               for k, v in valid_rates.items()}})
if failures:
    detail = "\n".join(f"  [{i + 1}] {m}" for i, m in enumerate(failures))
    raise AssertionError(f"NB 15 gate failed with {len(failures)} blocking issue(s):\n{detail}")
print()
print("NB 15 gate: PASSED")
print()
print(f"Reasoner {BEST_REASONER} / setting {BEST_SETTING} over {len(E1_ROSTERS)} rosters.")
print("NB 16 reads this notebook's winning configuration for the E6 decoding and prompt "
      "sensitivity study.")